# Team member(s):

If you export an `.ipynb` file from Kaggle, upon submitting your code on Canvas, please type the names of the team members here:

- Member 1: Roman Hunter

### UMKC Fall 2025 CS 5590/Math 5555 Final project advanced starter 2

This is the advanced starter code for the final project. Please walk it through.
You can download this starter as a notebook and upload it to Colab, or directly run it on Kaggle.
Do not attempt to run the code locally unless there is a GPU supporting CUDA.
All the given models are fairly large (> 10 million parameters) and we only about 30 GPU hours on Kaggle per week.
So please plan accordingly and do not start the final project in the last two weeks.

The end date of this competition is **Dec 20, 2025**.

The due date for submitting the final project report is **Dec 21, 2025**.

## Changes from the advanced starter:
- A K-fold validation strategy is added: monitoring the accuracy on 1-fold.
- A variant of SGD is implemented using Homework 7's problem 1 code with built-in inplace operations.
- A learning rate scheduler is added
- Tracking and plot the loss function values on train and validation set

## Things to try:
- Writing your own learning rate scheduler.
- Implement [RMSprop](https://optimization.cbe.cornell.edu/index.php?title=RMSProp#RMSProp) and [AdaDelta](https://optimization.cbe.cornell.edu/index.php?title=AdaGrad#AdaDelta) with momentum term using `torch.optim`'s `Optimizer` template class.

## GPU
On the right tab there is a Notebook config option page, there is GPU toggle under "Notebook options" tab, clicking the Accelerator drop-down menu will reveal your options. Please choose P100 when you plan to run the code. Kaggle has a Tesla P100 GPU which is better than Colab, but we have only 30 hours of GPU computing time reseting per week. Note this should be more than enough computing power to train the baseline model ResNet

In [ ]:
# imports
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt

# Use the ! (bang) prefix to run a shell command inside the notebook
!pip install torchinfo

In [ ]:
train = torch.load('/kaggle/input/2025-fall-cs-5590-math-5555-umkc/train.pt')
test = torch.load('/kaggle/input/2025-fall-cs-5590-math-5555-umkc/test.pt')

train_data, train_targets = train['data'], train['label']
test_data = test['data']

In [ ]:
# verifying the shapes
print(f"The shapes of train data, train targets, test data are\n {train_data.size()}, {train_targets.size()}, {test_data.size()}.")

The shapes of train data, train targets, test data are
 torch.Size([40000, 28, 28]), torch.Size([40000]), torch.Size([30000, 28, 28]).


The train set has data and targets given, yet test set has no targets. We have to train models based on the train data to achieve a high score on the leaderboard on the test set.

# Final Project: write an optimizer

We will have 3 given CNN models as follows, and we will write our own optimizer(s) using `torch.optim` interface to optimize the given 3 models. You are free to use any techniques learned in the class to optimize the model. The goal is to optimize the model such that the model(s) trained performs good in the test set with unknown targets.


## Models:
The 3 fixed models we can use are the famous first state-of-art image recognition CNN ResNet: ResNet 18, ResNet 34, and ResNet 509\ [https://pytorch.org/hub/pytorch_vision_resnet/](https://pytorch.org/hub/pytorch_vision_resnet/). Their settings are in the next cell.
The `__init__()` has two parameters:
- `block`: the first type is the `BasicBlock` used for building ResNet, defined in the file (`torchvision.models.resnet`). The other type is `Bottleneck` block, which has a more agressive dimension reduction mechanics.
- `[2, 2, 2, 2]`: Each number denotes the number of `Bottleneck` or `BasicBlock` modules in a "stage". It depends on how deep you want the networks to be. For example, for ResNet18, it is `[2, 2, 2, 2]`; for ResNet34, it is `[3, 4, 6, 3]`.

The full code of ResNet is here: [https://github.com/pytorch/vision/blob/master/torchvision/models/resnet.py](https://github.com/pytorch/vision/blob/master/torchvision/models/resnet.py). We are just gonna import the `torchvision` implementation, and change the first convolutional layer to accept single channel input, and the stride of the first convolutional layer from 2 to 1.


## Validation:

Since you do not have access to the target of the test set, to cross-validate your model, you should split the training samples to two sets using the techniques learned in class: one is training dataset, the other is your validation dataset (so that you have access to the targets of this set). This routine is usually used for Kaggle competitions and in the real world model deployment to ensure the trained model has generalizability to the unseen data. In this starter code, **K-fold** is used (yet only trained on 1 fold).

## Hyperparameter tuning:
Aside from writing our own optimizer, there are quite a few hyperparameter for us to tune to achieve better performance:
- learning rate and/or with a scheduler
- dropout rate
- early stop
- other validation strategy based tuning
It is okay to use autoML package (H2O, Rapids, Optuna, fast.ai) if you know how to write a wrapper of them using the given model.

In [ ]:
import torch.nn as nn
from torchvision.models.resnet import ResNet, BasicBlock

class MyResNet(ResNet):
    def __init__(self,
                 block=BasicBlock,
                 layers=[2, 2, 2, 2],
                 num_classes=10):
        super(MyResNet, self).__init__(block,
                                       layers,
                                       num_classes=num_classes)

        # 1. Optimize conv1 for 28x28 (reduced kernel and padding)
        self.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)

        # 2. Remove the MaxPool layer
        # Setting it to nn.Identity() effectively removes it while keeping the forward flow intact
        self.maxpool = nn.Identity()

        self.dropout = nn.Dropout(p=0.3)  # Initialize Dropout


    def set_dropout(self, rate): # Change 'model' to 'self'
            for module in self.modules(): # Change 'model' to 'self'
                if isinstance(module, torch.nn.Dropout):
                    module.p = rate
            print(f"🔄 Dropout rate updated to: {rate}")

    def forward(self, x):
        # Override forward to inject dropout before the final fully connected layer (self.fc)
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        x = self.dropout(x) # Apply dropout here to help close generalization gap
        x = self.fc(x)

        return x

# Initialization
resnet18_config = {
    "block": BasicBlock,
    "layers": [2, 2, 2, 2]
}

model = MyResNet(**resnet18_config)

In [ ]:
from torchinfo import summary # similar function to tensorflow/keras summary for torch

In [ ]:
batch_size = 64
summary(model, input_size=(batch_size, 1, 28, 28), device='cpu'); # (color_channel, 28, 28)

A batch size of 1024 of float precision training needs about 4GB of RAM for the GPU.

# Prepare the train loader

In [ ]:
USE_HOLDOUT = True  # Set to False when you want to train on the FULL dataset

if USE_HOLDOUT:
    from sklearn.model_selection import train_test_split
    # Split 10% off the top. This data is "hidden" from the K-Fold entirely.
    train_data_kf, holdout_data, train_targets_kf, holdout_targets = train_test_split(
        train_data, train_targets,
        test_size=0.10,
        stratify=train_targets,
        random_state=42
    )
    print(f"Locked away {len(holdout_data)} images for the final gold-standard test.")
else:
    # Use everything for training
    train_data_kf, train_targets_kf = train_data, train_targets
    print("Training on the full dataset (No holdout).")

In [ ]:
def zeropower_via_newtonschulz5(G, steps: int):
    """
    Newton-Schulz iteration to compute the zeroth power / orthogonalization of G. We opt to use a
    quintic iteration whose coefficients are selected to maximize the slope at zero. For the purpose
    of minimizing steps, it turns out to be empirically effective to keep increasing the slope at
    zero even beyond the point where the iteration no longer converges all the way to one everywhere
    on the interval. This iteration therefore does not produce UV^T but rather something like US'V^T
    where S' is diagonal with S_{ii}' ~ Uniform(0.5, 1.5), which turns out not to hurt model
    performance at all relative to UV^T, where USV^T = G is the SVD.
    """
    assert G.ndim >= 2 # batched Muon implementation by @scottjmaddox, and put into practice in the record by @YouJiacheng
    a, b, c = (3.4445, -4.7750,  2.0315)
    X = G.bfloat16()
    if G.size(-2) > G.size(-1):
        X = X.mT

    # Ensure spectral norm is at most 1
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    # Perform the NS iterations
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * A @ A # quintic computation strategy adapted from suggestion by @jxbz, @leloykun, and @YouJiacheng
        X = a * X + B @ X

    if G.size(-2) > G.size(-1):
        X = X.mT
    return X



# Final project: write our own optimizer

In the following cells, we use the `torch.optim` interface to build our own optimizer.
The following code is simplified from an early version of [SGD in PyTorch 1.x](https://docs.pytorch.org/docs/1.9.0/optim.html?highlight=sgd#torch.optim.SGD)

Please refer to the final project page to see requirements of the implementation of a customized optimizer.

In [ ]:
import torch
from torch.optim import Optimizer
import math

class MuonAdaDeltaLookahead(Optimizer):
    def __init__(self, params, muon_lr=1e-3, rho=0.95, eps=1e-6, k=5, ns_steps=5):
        defaults = dict(lr=1.0, muon_lr=muon_lr, rho=rho, eps=eps, k=k, ns_steps=ns_steps)
        super().__init__(params, defaults)
        self.fast_magnitude = 0.0
        self.slow_magnitude = 0.0

    @torch.no_grad()
    def step(self, closure=None):
        total_fast = 0.0
        total_slow = 0.0
        count_fast = 0
        count_slow = 0

        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                state = self.state[p]
                if len(state) == 0:
                    state['step'] = 0
                    state['square_avg'] = torch.zeros_like(p)
                    state['acc_deltas'] = torch.zeros_like(p)
                    state['muon_buf'] = torch.zeros_like(p)

                state['step'] += 1
                grad = p.grad
                rho, eps = group['rho'], group['eps']

                # --- FAST PATH (AdaDelta) ---
                state['square_avg'].mul_(rho).addcmul_(grad, grad, value=1 - rho)
                std_grads = state['square_avg'].add(eps).sqrt()
                std_deltas = state['acc_deltas'].add(eps).sqrt()

                # The actual delta
                update_step = grad.mul(std_deltas).div(std_grads).mul(group['lr'])
                p.data.add_(update_step, alpha=-1.0)
                state['acc_deltas'].mul_(rho).addcmul_(update_step, update_step, value=1 - rho)

                total_fast += update_step.norm(2).item()
                count_fast += 1

                # --- SLOW PATH (Muon) ---
                if state['step'] % group['k'] == 0 and p.ndim >= 2:
                    muon_buf = state['muon_buf']
                    muon_buf.mul_(0.95).add_(grad, alpha=1 - 0.95)

                    update = muon_buf.clone()
                    if update.ndim == 4: update = update.view(len(update), -1)
                    update = zeropower_via_newtonschulz5(update, steps=group['ns_steps'])
                    update *= max(1, update.size(-2) / update.size(-1))**0.5

                    # Apply Muon Update
                    actual_muon_update = update.reshape(p.shape) * group['muon_lr']
                    p.data.add_(actual_muon_update, alpha=-1.0)

                    total_slow += actual_muon_update.norm(2).item()
                    count_slow += 1

        # Store averages for logging
        self.fast_magnitude = total_fast / count_fast if count_fast > 0 else 0
        self.slow_magnitude = total_slow / count_slow if count_slow > 0 else 0
        return loss

## Final project: add a schedular

In [ ]:
from torch.optim.lr_scheduler import _LRScheduler

class MuonAdaDeltaScheduler(_LRScheduler):
    def __init__(self, optimizer, last_epoch=-1):
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        """
        In AdaDelta, the 'lr' group usually stays at 1.0.
        We track 'muon_lr' as the meaningful metric for logging.
        """
        return [group['muon_lr'] for group in self.optimizer.param_groups]

    def step_lr(self, factor=0.1):
        """
        Manually scale the Muon learning rate and the AdaDelta global coefficient.
        """
        for group in self.optimizer.param_groups:
            # 1. Scale the Muon update magnitude
            group['muon_lr'] *= factor

            # 2. Scale the AdaDelta 'lr'. Even though AdaDelta is adaptive,
            # this 'lr' acts as a global multiplier on its final output.
            group['lr'] *= factor

            print(f"Decayed rates: Muon LR -> {group['muon_lr']:.3e}, AdaDelta Coeff -> {group['lr']:.3e}")

# Training

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device) # should be cuda if GPU is enabled

cuda


In [ ]:
from tqdm.auto import tqdm

# --- ENSEMBLE HYPERPARAMETERS (AdaDelta + Muon) ---
# AdaDelta doesn't use a standard LR, but we use 1.0 as the base coefficient.
adadelta_coeff = 1.0
muon_lr = 1e-3        # Muon's spectral norm LR
rho = 0.95            # Adadelta/Muon decay rate (equivalent to beta)
k_interval = 30       # Sync every 30 steps
ns_steps = 5          # Newton-Schulz iterations
weight_decay = 0.0   # Optional: apply to non-orthogonalized weights
# --------------------------------

# 1. Initialize the Ensemble Optimizer
# We pass parameters for both AdaDelta and Muon
optimizer = MuonAdaDeltaLookahead(
    model.parameters(),
    muon_lr=muon_lr,
    rho=rho,          # Used for both Adadelta accumulation and Muon momentum
    eps=1e-6,
    k=k_interval,
    ns_steps=ns_steps
    # Note: If your MuonAdaDelta class supports weight_decay, add it here
)

# 2. Scheduler
# Since AdaDelta is self-regulating, we primarily use the scheduler
# to "throttle" the Muon updates and the AdaDelta global coefficient.
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',       # 'max' because you are passing acc_last_epoch
    factor=0.5,
    patience=2
)

# 3. Patch the optimizer for Scheduler compatibility
# We set 'lr' to the Adadelta coefficient.
# ReduceLROnPlateau will now modify 'lr' and 'muon_lr' simultaneously
# if you use the custom sync logic in the training loop.
for group in optimizer.param_groups:
    group['lr'] = adadelta_coeff

loss_func = torch.nn.CrossEntropyLoss(label_smoothing=0.1)
model.to(device);

In [ ]:
import os
from sklearn.model_selection import StratifiedKFold

# --- CONFIGURATION ---
k_folds = 5
epochs = 30
batch_size = 64
initial_muon_lr = 1e-3
skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)

# Global results tracking
all_fold_best_accs = []

# Assuming train_data and train_targets are your loaded Kuzushiji-MNIST tensors
for fold, (train_idx, val_idx) in enumerate(skf.split(train_data, train_targets)):
    model = MyResNet(**resnet18_config).to(device)
    model.set_dropout(0.4)
    print(f"\n{'='*25} STARTING FOLD {fold+1}/{k_folds} {'='*25}")

    # 1. DATA LOADERS FOR THIS FOLD
    X_tr, y_tr = train_data[train_idx][:,None,:,:].float()/255.0, train_targets[train_idx].long()
    X_val, y_val = train_data[val_idx][:,None,:,:].float()/255.0, train_targets[val_idx].long()

    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

    optimizer = MuonAdaDeltaLookahead(
        model.parameters(),
        muon_lr=initial_muon_lr,
        rho=0.9,           # Or use your 'rho' variable
        eps=1e-6,
        k=5,               # Or use your 'k_interval' variable
        ns_steps=5         # Or use your 'ns_steps' variable
    )

    # Since this version doesn't have a top-level 'lr' to decay,
    # we update muon_lr directly in the scheduler step later.

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=0.5)
    loss_func = nn.CrossEntropyLoss(label_smoothing=0.1)

    fold_acc_max = 0.0

    # 3. EPOCH LOOP
    for epoch in range(epochs):
        # --- DYNAMIC REGULARIZATION DECAY ---
        if epoch == 15: # Accelerated schedule for 30 epochs
            model.set_dropout(0.2)
            loss_func.label_smoothing = 0.05
        elif epoch == 25:
            model.set_dropout(0.1)
            loss_func.label_smoothing = 0.02

        # --- TRAINING PHASE ---
        model.train()
        loss_vals, train_acc_accum = [], []

        for x, targets in train_loader:
            x, targets = x.to(device), targets.to(device)

            outputs = model(x)
            loss = loss_func(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_vals.append(loss.item())
            train_acc_accum.append((outputs.argmax(-1) == targets).float().mean().item())

        # --- VALIDATION PHASE ---
        model.eval()
        acc_on_valid = []
        with torch.no_grad():
            for x_v, y_v in val_loader:
                x_v, y_v = x_v.to(device), y_v.to(device)
                yhat = model(x_v)
                acc_on_valid.append((yhat.argmax(-1) == y_v).float().mean().item())

        avg_val_acc = np.mean(acc_on_valid)

        # --- SYNC MUON LR WITH ADADELTA DECAY ---
        current_lr = optimizer.param_groups[0]['lr']
        for group in optimizer.param_groups:
            group['muon_lr'] = initial_muon_lr * current_lr

        # Scheduler Step
        scheduler.step(avg_val_acc)

        print(f"Fold {fold+1} E{epoch+1} | Val Acc: {avg_val_acc:.4f} | MuonLR: {optimizer.param_groups[0]['muon_lr']:.2e}")

        # Save Best for this fold
        if avg_val_acc > fold_acc_max:
            fold_acc_max = avg_val_acc
            torch.save(model.state_dict(), f'muon_fold_{fold+1}.pt')

    all_fold_best_accs.append(fold_acc_max)
    print(f"✅ Fold {fold+1} Best: {fold_acc_max:.4f}")

print(f"\n🏆 Final Muon Ensemble CV Score: {np.mean(all_fold_best_accs):.4f}")

🔄 Dropout rate updated to: 0.4

========================= STARTING FOLD 1/5 =========================
Fold 1 E1 | Val Acc: 0.9805 | MuonLR: 1.00e-03
Fold 1 E2 | Val Acc: 0.9855 | MuonLR: 1.00e-03
Fold 1 E3 | Val Acc: 0.9862 | MuonLR: 1.00e-03
Fold 1 E4 | Val Acc: 0.9764 | MuonLR: 1.00e-03
Fold 1 E5 | Val Acc: 0.9925 | MuonLR: 1.00e-03
Fold 1 E6 | Val Acc: 0.9925 | MuonLR: 1.00e-03
Fold 1 E7 | Val Acc: 0.9926 | MuonLR: 1.00e-03
Fold 1 E8 | Val Acc: 0.9921 | MuonLR: 1.00e-03
Fold 1 E9 | Val Acc: 0.9919 | MuonLR: 1.00e-03
Fold 1 E10 | Val Acc: 0.9919 | MuonLR: 1.00e-03
Fold 1 E11 | Val Acc: 0.9916 | MuonLR: 5.00e-04
Fold 1 E12 | Val Acc: 0.9924 | MuonLR: 5.00e-04
Fold 1 E13 | Val Acc: 0.9936 | MuonLR: 5.00e-04
Fold 1 E14 | Val Acc: 0.9932 | MuonLR: 5.00e-04
Fold 1 E15 | Val Acc: 0.9932 | MuonLR: 5.00e-04
🔄 Dropout rate updated to: 0.2
Fold 1 E16 | Val Acc: 0.9928 | MuonLR: 5.00e-04
Fold 1 E17 | Val Acc: 0.9928 | MuonLR: 2.50e-04
Fold 1 E18 | Val Acc: 0.9935 | MuonLR: 2.50e-04
Fold 1 E19 |

In [ ]:
def evaluate_ensemble(models_paths, val_loader, device):
    all_probs = []

    # We need a fresh model instance to load weights into
    eval_model = MyResNet(**resnet18_config).to(device)
    eval_model.eval()

    with torch.no_grad():
        for path in models_paths:
            print(f"⌛ Loading weights from {path}...")
            eval_model.load_state_dict(torch.load(path))

            fold_probs = []
            targets_list = []

            for x, y in val_loader:
                x = x.to(device)
                logits = eval_model(x)
                # Softmax turns logits into probabilities (0 to 1)
                probs = torch.softmax(logits, dim=-1)
                fold_probs.append(probs.cpu())
                targets_list.append(y.cpu())

            all_probs.append(torch.cat(fold_probs))

        # Stack the 5 models' probabilities and average them
        # Shape: [5, Num_Images, 10_Classes]
        stacked_probs = torch.stack(all_probs)
        avg_probs = torch.mean(stacked_probs, dim=0)

        # Get final predictions
        final_preds = avg_probs.argmax(dim=-1)
        final_targets = torch.cat(targets_list)

        ensemble_acc = (final_preds == final_targets).float().mean()
        return ensemble_acc.item()

# Example usage after your K-fold loop finishes:
# muon_paths = [f'muon_fold_{i+1}.pt' for i in range(5)]
# ensemble_score = evaluate_ensemble(muon_paths, val_loader, device)
# print(f"🔥 Final Ensemble Validation Accuracy: {ensemble_score:.4f}")

## Plot the train/valid loss as a function of iterations

# Predict the solution on the test set

Evaluate the trained model and generate the `preds` variable, which is the index of the maximum entry of the 10 outputs for each sample.

In [ ]:
model.load_state_dict(torch.load('./my_model.pt', map_location=device))
model.eval(); # this cannot be omitted due to dropout

In [ ]:
test = TensorDataset(test_data[:,None,:,:].float())
test_loader = DataLoader(test, batch_size = batch_size, shuffle = False, drop_last=False)

y_preds = []
for x in tqdm(test_loader):
    with torch.no_grad():
        x = x[0].to(device)
        y_pred = model(x)
        y_preds.append(y_pred.cpu().detach().numpy())

y_preds = np.concatenate(y_preds)
preds = y_preds.argmax(axis=-1).astype(int)
print(preds.shape) # should be 30,000

# Export the solutions and submit to Kaggle

The result predicted by your model can be named to `preds`, and be exported to a `.csv` file using the following cell. The `y_pred` should be of a dimension `(30000,)` numpy array.

Rename the resulting `solutions-yournames.csv` by replacing `yournames` by your team members' initials connected by hyphen and then save the notebook on Kaggle.

Then click the submit button below, the solution can be submitted directly from the output tab of a committed notebook (click `Save Version` blue button in the interactive kernel window).

In [ ]:
len_test = test_data.size(0)
solutions = np.zeros((test_data.size(0), 2))
solutions[:,0] = np.arange(1,len_test+1)
solutions[:,1] = preds
solutions = solutions.astype(int)
np.savetxt("solutions-rh-MuonAdaDelta3.csv", solutions,
           fmt='%s', header = 'Id,Category', delimiter = ',', comments='')